# **Model Trigonometri dalam Sistem Navigasi Astronomi untuk Peningkatan Akurasi Posisi Satelit**

In [9]:
import math

class Perhitungan():
	def __init__(self, local_sidereal_time, suhu_lingkungan, 
			  tekanan_udara, azimut, altitude_elevasi, ketinggian_orbit_satelit):
		self.lokasi_pengamatan = 3.5275
		self.local_sidereal_time = local_sidereal_time
		self.suhu_lingkungan = suhu_lingkungan
		self.tekanan_udara = tekanan_udara
		self.azimut = azimut
		self.altitude_elevasi = altitude_elevasi
		self.jari_jari_bumi = 6378.137
		self.ketinggian_orbit_satelit = ketinggian_orbit_satelit
		self.jari_jari_orbit_satelit = self.jari_jari_bumi + self.ketinggian_orbit_satelit

		self.kalkulasi()		

	def kalkulasi(self):
		def sin(degree):
			return math.sin(math.radians(degree))

		def cos(degree):
			return math.cos(math.radians(degree))

		def tan(degree):
			return math.tan(math.radians(degree))	

		def solve_abc(a, b, c):
			d = b**2 - 4*a*c
			if d >= 0:
				sol1 = (-b + math.sqrt(d)) / 2*a
				sol2 = (-b - math.sqrt(d)) / 2*a
				return max(sol1, sol2)

		def spatial_error():
			delta_east = round(self.toposentrik_east_nampak - self.toposentrik_east, 6)
			delta_north = round(self.toposentrik_north_nampak - self.toposentrik_north, 6)
			delta_up = round(self.toposentrik_up_nampak - self.toposentrik_up, 6)

			print(f"Delta east : {delta_east} \nDelta north : {delta_north} \nDelta up : {delta_up}")

			return math.sqrt(delta_east**2 + delta_north**2 + delta_up**2)

		self.faktor_suhu_tekanan = (self.tekanan_udara / 1010) * (283 / (273 + self.suhu_lingkungan))
		self.faktor_suhu_tekanan = round(self.faktor_suhu_tekanan, 6)
		print(f"Faktor suhu tekanan : {self.faktor_suhu_tekanan}")

		self.argumen_tan = self.altitude_elevasi + (10.3 / (self.altitude_elevasi + 5.11))
		self.argumen_tan = round(self.argumen_tan, 6)
		print(f"Argumen tan : {self.argumen_tan}")

		self.nilai_refraksi = self.faktor_suhu_tekanan * (1.02 / tan(self.argumen_tan)) / 60
		self.nilai_refraksi = round(self.nilai_refraksi, 6)
		print(f"Nilai refraksi : {self.nilai_refraksi}")

		self.a_sebenarnya = self.altitude_elevasi - self.nilai_refraksi
		self.a_sebenarnya = round(self.a_sebenarnya, 6)
		print(f"Nilai a sebanarnya : {self.a_sebenarnya}")

		self.deklinasi = math.degrees(math.asin(sin(self.lokasi_pengamatan) * sin(self.a_sebenarnya) + cos(self.lokasi_pengamatan) * cos(self.a_sebenarnya) * cos(self.azimut)))
		self.deklinasi = round(self.deklinasi,6)
		print(f"Deklinasi : {self.deklinasi}")

		self.hour_angle = math.degrees(math.acos((sin(self.a_sebenarnya) - sin(self.lokasi_pengamatan)*sin(self.deklinasi)) / (cos(self.lokasi_pengamatan) * cos(self.deklinasi))))
		self.hour_angle = round(360 - self.hour_angle, 6)
		print(f"Hour angle : {self.hour_angle}")

		self.asensio_rekta = self.local_sidereal_time - self.hour_angle + 360
		print(f"Asensio rekta : {self.asensio_rekta}")	

		self.derivasi_jarak_toposentrik_satelit_nampak = solve_abc(1, 2 * self.jari_jari_bumi * math.sin(math.radians(self.altitude_elevasi)), self.jari_jari_bumi**2 - self.jari_jari_orbit_satelit**2)	
		self.derivasi_jarak_toposentrik_satelit_nampak = round(self.derivasi_jarak_toposentrik_satelit_nampak, 6)
		print(f"Deviasi jarak toposentrik nampak : {self.derivasi_jarak_toposentrik_satelit_nampak}")

		self.toposentrik_east_nampak = self.derivasi_jarak_toposentrik_satelit_nampak * cos(self.altitude_elevasi) * sin(self.azimut)
		self.toposentrik_east_nampak = round(self.toposentrik_east_nampak, 6)
		print(f"Toposentrik east nampak : {self.toposentrik_east_nampak}")

		self.toposentrik_north_nampak = self.derivasi_jarak_toposentrik_satelit_nampak * cos(self.altitude_elevasi) * cos(self.azimut)
		self.toposentrik_north_nampak = round(self.toposentrik_north_nampak, 6)
		print(f"Toposentrik north nampak : {self.toposentrik_north_nampak}")

		self.toposentrik_up_nampak = self.derivasi_jarak_toposentrik_satelit_nampak * sin(self.altitude_elevasi)
		self.toposentrik_up_nampak = round(self.toposentrik_up_nampak, 6)
		print(f"Toposentrik up nampak : {self.toposentrik_up_nampak}")		

		self.derivasi_jarak_toposentrik_satelit = solve_abc(1, 2 * self.jari_jari_bumi * math.sin(math.radians(self.a_sebenarnya)), self.jari_jari_bumi**2 - self.jari_jari_orbit_satelit**2)	
		self.derivasi_jarak_toposentrik_satelit = round(self.derivasi_jarak_toposentrik_satelit, 6)
		print(f"Deviasi jarak toposentrik : {self.derivasi_jarak_toposentrik_satelit}")

		self.toposentrik_east = self.derivasi_jarak_toposentrik_satelit * cos(self.a_sebenarnya) * sin(self.azimut)
		self.toposentrik_east = round(self.toposentrik_east, 6)
		print(f"Toposentrik east : {self.toposentrik_east}")

		self.toposentrik_north = self.derivasi_jarak_toposentrik_satelit * cos(self.a_sebenarnya) * cos(self.azimut)
		self.toposentrik_north = round(self.toposentrik_north, 6)
		print(f"Toposentrik north : {self.toposentrik_north}")

		self.toposentrik_up = self.derivasi_jarak_toposentrik_satelit * sin(self.a_sebenarnya)
		self.toposentrik_up = round(self.toposentrik_up, 6)
		print(f"Toposentrik up : {self.toposentrik_up}")

		self.error = spatial_error()
		self.error = round(self.error, 6)
		print(f"Spatial error : {self.error}")

In [10]:
# 09:53:57
Perhitungan(local_sidereal_time=114.615956, 
            suhu_lingkungan=30, 
            tekanan_udara=1012, 
            azimut=76.66, 
            altitude_elevasi=51.21,
            ketinggian_orbit_satelit=489.45)

Faktor suhu tekanan : 0.935843
Argumen tan : 51.392884
Nilai refraksi : 0.012703
Nilai a sebanarnya : 51.197297
Deklinasi : 11.084661
Hour angle : 321.587123
Asensio rekta : 153.02883299999996
Deviasi jarak toposentrik nampak : 614.109004
Toposentrik east nampak : 374.339016
Toposentrik north nampak : 88.765981
Toposentrik up nampak : 478.665615
Deviasi jarak toposentrik : 614.206432
Toposentrik east : 374.501672
Toposentrik north : 88.804551
Toposentrik up : 478.656234
Delta east : -0.162656 
Delta north : -0.03857 
Delta up : 0.009381
Spatial error : 0.167429


In [11]:
# 09:56:50
Perhitungan(local_sidereal_time=115.342941, 
            suhu_lingkungan=30, 
            tekanan_udara=1012, 
            azimut=76.5, 
            altitude_elevasi=51.91,
            ketinggian_orbit_satelit=489.56)

Faktor suhu tekanan : 0.935843
Argumen tan : 52.090638
Nilai refraksi : 0.012389
Nilai a sebanarnya : 51.897611
Deklinasi : 11.080941
Hour angle : 322.307902
Asensio rekta : 153.03503899999998
Deviasi jarak toposentrik nampak : 608.960248
Toposentrik east nampak : 365.286975
Toposentrik north nampak : 87.697644
Toposentrik up nampak : 479.27772
Deviasi jarak toposentrik : 609.052317
Toposentrik east : 365.44298
Toposentrik north : 87.735097
Toposentrik up : 479.268928
Delta east : -0.156005 
Delta north : -0.037453 
Delta up : 0.008792
Spatial error : 0.160679


In [12]:
# 10:00:09
Perhitungan(local_sidereal_time=116.1702, 
            suhu_lingkungan=30, 
            tekanan_udara=1012, 
            azimut=76.3, 
            altitude_elevasi=52.716,
            ketinggian_orbit_satelit=470.24)

Faktor suhu tekanan : 0.935843
Argumen tan : 52.894121
Nilai refraksi : 0.012035
Nilai a sebanarnya : 52.703965
Deklinasi : 11.080174
Hour angle : 323.138971
Asensio rekta : 153.03122899999997
Deviasi jarak toposentrik nampak : 579.696094
Toposentrik east nampak : 341.169502
Toposentrik north nampak : 83.16816
Toposentrik up nampak : 461.23095
Deviasi jarak toposentrik : 579.779319
Toposentrik east : 341.312614
Toposentrik north : 83.203047
Toposentrik up : 461.223385
Delta east : -0.143112 
Delta north : -0.034887 
Delta up : 0.007565
Spatial error : 0.147497


In [13]:
# 11:12:17
Perhitungan(local_sidereal_time=134.252907, 
            suhu_lingkungan=31, 
            tekanan_udara=1011, 
            azimut=67.03, 
            altitude_elevasi=69.89,
            ketinggian_orbit_satelit=470.38)

Faktor suhu tekanan : 0.931843
Argumen tan : 70.027333
Nilai refraksi : 0.005757
Nilai a sebanarnya : 69.884243
Deklinasi : 11.053983
Hour angle : 341.177689
Asensio rekta : 153.075218
Deviasi jarak toposentrik nampak : 498.633325
Toposentrik east nampak : 157.848155
Toposentrik north nampak : 66.905048
Toposentrik up nampak : 468.233774
Deviasi jarak toposentrik : 498.650262
Toposentrik east : 157.896834
Toposentrik north : 66.925681
Toposentrik up : 468.23245
Delta east : -0.048679 
Delta north : -0.020633 
Delta up : 0.001324
Spatial error : 0.052888


In [14]:
# 11:13:09
Perhitungan(local_sidereal_time=134.470166, 
            suhu_lingkungan=31, 
            tekanan_udara=1011, 
            azimut=66.81, 
            altitude_elevasi=70.08,
            ketinggian_orbit_satelit=470.02)

Faktor suhu tekanan : 0.931843
Argumen tan : 70.216986
Nilai refraksi : 0.005698
Nilai a sebanarnya : 70.074302
Deklinasi : 11.057337
Hour angle : 341.385966
Asensio rekta : 153.0842
Deviasi jarak toposentrik nampak : 497.697932
Toposentrik east nampak : 155.869016
Toposentrik north nampak : 66.773347
Toposentrik up nampak : 467.920294
Deviasi jarak toposentrik : 497.714497
Toposentrik east : 155.916979
Toposentrik north : 66.793894
Toposentrik up : 467.919001
Delta east : -0.047963 
Delta north : -0.020547 
Delta up : 0.001293
Spatial error : 0.052195
